In [ ]:
%stop_session

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

### Setup

In [ ]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE         = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO       = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE         = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO       = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD                    = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV                = f"{BASE}/gold/gmv_daily_by_subsidiary"

In [ ]:
from pyspark.sql.functions import col, lit, sum as spark_sum, countDistinct, current_timestamp

DQ_PATH = "s3://data-lake-case-hotmart/gold/dq_gmv_run_metrics"
QUARANTINE_PATH = "s3://data-lake-case-hotmart/gold/quarantine_missing_items"

# Base "as-of" corrente do snapshot (mesma fonte do gvm_gold)
df_gvm_snapshot = spark.table("gvm_gold").where(col("snapshot_date") == lit(df_snapshot_date))

# 1) Quarentena: compras faturadas sem item (não entram no GMV)
df_quarantine_missing_item = (
    df_gvm_snapshot
    .where(col("release_date").isNotNull())
    .where(col("product_id").isNull())  # sem item
    .select(
        "snapshot_date",
        "purchase_id",
        "buyer_id",
        "prod_item_id",
        "order_date",
        "release_date",
        "producer_id",
        "subsidiary",
        "transaction_date",
        "snapshot_datetime"
    )
)

df_quarantine_missing_item.write.format("delta") \
    .mode("append") \
    .partitionBy("snapshot_date") \
    .save(QUARANTINE_PATH)


In [ ]:
# 2) Métricas DQ do run
df_dq_metrics = (
    df_gvm_snapshot.agg(
        countDistinct("purchase_id").alias("total_purchases_asof"),
        countDistinct(F.when(col("release_date").isNotNull(), col("purchase_id"))).alias("succeded_transactions"),
        F.sum(F.when(col("product_id").isNull() & col("release_date").isNotNull(), 1).otherwise(0)).alias("missing_product_item_cnt"),
        F.sum(F.when(col("subsidiary").isNull(), 1).otherwise(0)).alias("missing_extra_info_cnt"),
        F.sum(F.when(col("subsidiary") == "UNKNOWN", 1).otherwise(0)).alias("unknown_subsidiary_cnt"),
        F.count("*").alias("gvm_rows"),
        spark_sum(col("item_quantity") * col("purchase_value")).alias("gmv_total_value")
    )
    .withColumn("snapshot_date", lit(df_snapshot_date).cast("date"))
    .withColumn("snapshot_datetime", current_timestamp())
)

df_dq_metrics.write.format("delta") \
    .mode("append") \
    .partitionBy("snapshot_date") \
    .save(DQ_PATH)